# Perceptró multi-capa a PyTorch

## Introdicció a Pytorch

PyTorch és una llibreria d'aprenentatge profund desenvolupada per Meta que permet construir i entrenar xarxes neuronals de manera flexible i eficient. A diferència de scikit-learn, que abstreu completament el procés d'entrenament, PyTorch dona control total sobre cada pas del training loop, cosa que el fa especialment adequat per a arquitectures complexes com RNNs, CNNs o Transformers. La diferència fonamental amb scikit-learn és que a PyTorch som nosaltres qui escrivim explícitament el procés d'entrenament: allò que a scikit-learn fa `model.fit()` de manera automàtica, a PyTorch ho haurem d'implementar nosaltres pas a pas.

En aquesta secció aprendrem a construir el nostre MLP per, d'aquesta manera descobrir el funcionament de la llibreria [Pytorch](https://pytorch.org/).

## Construcció d'un MLP


Emprar l'estructura `nn.Sequential` és la manera més senzilla de construir una xarxa a PyTorch. Permet definir l'arquitectura com una seqüència ordenada de capes, de manera similar a com especifiquem `hidden_layer_sizes` a scikit-learn.

En el següent exemple veiem com definim un MLP pel problema que hem vist a la secció anterior. En aquest cas hem de definir nosaltres tant el nombre d'elements de la capa d'entrada com de la capa de sortida. En aquest cas no diferenciem entre problemes de classificació i regressió, la definició de la sortida de la xarxa i les funcions d'activació i de pèrdua que emprém, seran els que definiran la seva funcionalitat.

```python
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(10, 64),   # capa d'entrada → primera capa oculta
    nn.ReLU(),
    nn.Linear(64, 32),   # primera capa oculta → segona capa oculta
    nn.ReLU(),
    nn.Linear(32, 3)     # segona capa oculta → capa de sortida (3 classes)
)
```

Cada `nn.Linear(in, out)` defineix una capa totalment connectada amb els pesos $W$ i biaixos $b$ corresponents. Les funcions d'activació s'afegeixen explícitament entre capes, a diferència de scikit-learn on s'especifiquen amb el paràmetre `activation`.

---

## Exemple complet


A continuació explicarem el funcionament del codi necessari per entrenar i avaluar un MLP, solventarem el mateix problema que en la secció anterior per tal de poder observar les semblances i les diferències entre les dues llibreries.


In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Generació i separació de dades (igual que amb scikit-learn)
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_classes=3,
    n_informative=6,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Conversió de matrius Numpy tensors de PyTorch: Aquesta passa és obligatòria
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test,  dtype=torch.long)

# Definició del model (mateixa arquitectura que MLPClassifier)
model = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 3)
)

# Definició de la funció de pèrdua i de l'optimitzador
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # El ratio d'aprenentatge es defineix aquí

# Bucle d'entrenament
epochs = 200
for epoch in range(epochs):
    model.train()

    # 1. Pas cap endavant
    y_pred = model(X_train)

    # 2. Càlcul de la pèrdua
    loss = criterion(y_pred, y_train)

    # 3. Backward pass
    optimizer.zero_grad()
    loss.backward()

    # 4. Actualització de pesos
    optimizer.step()
    # Mostrar resultats parcials
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}")

# Avaluació del model
model.eval()
with torch.no_grad():
    y_pred_test = model(X_test)
    predicted   = torch.argmax(y_pred_test, dim=1)
    accuracy    = (predicted == y_test).float().mean()
    print(f"\nAccuracy: {accuracy:.4f}")


Aspectes a tenir en compte d'aquest codi que el fan molt diferents a l'entrenament amb Scikit:

1. `optimizer.zero_grad()` és necessari perquè PyTorch acumula gradients per defecte. Això significa que si no els resetegem, els gradients de la iteració anterior se sumen als de la iteració actual, donant lloc a actualitzacions de pesos incorrectes.
2. `model.eval()` i `torch.no_grad()` desactiven el càlcul de gradients durant l'avaluació, estalviant memòria.
3. `nn.CrossEntropyLoss` ja incorpora la funció Softmax internament, per això la capa de sortida no té funció d'activació.


## Tensors
Com hauràs observat, en el codi hem transformat les matrius `ndarray` de Numpy al tipus de dades `tensor`. Un tensor és l'estructura de dades bàsica de PyTorch, equivalent als arrays de NumPy però amb la capacitat addicional d'executar-se en GPU i de calcular gradients automàticament. Un escalar, un vector, una matriu o qualsevol array n-dimensional es representen com a tensors.

Podem crear tensors de manera manual, encara que no serà una cosa molt habitual:
```python
import torch

# A partir d'una llista
t = torch.tensor([1.0, 2.0, 3.0])

# Tensors especials
torch.zeros(3, 4)      # matriu 3x4 de zeros
torch.ones(3, 4)       # matriu 3x4 d'uns
torch.rand(3, 4)       # matriu 3x4 de valors aleatoris entre 0 i 1
torch.randn(3, 4)      # matriu 3x4 de valors aleatoris amb distribució normal
```

### Tipus de dades (dtype)
El tipus de dades és important perquè PyTorch és estricte: operacions entre tensors de tipus diferents generen errors.


| `dtype` | Descripció | Ús típic |
|---------|------------|----------|
| `torch.float32` | Decimal 32 bits | Features, pesos de la xarxa |
| `torch.float64` | Decimal 64 bits | Precisió alta (poc freqüent) |
| `torch.long` | Enter 64 bits | Etiquetes de classificació |
| `torch.bool` | Booleà | Màscares |

```python
t = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)
print(t.dtype)   # torch.float32
```

### Conversió entre NumPy i PyTorch

Transformar una estructura de dades entre aquestes dues llibreries és directe i senzill, només cal conèixer una única operació:

```python
import numpy as np

# NumPy → PyTorch
array = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(array)

# PyTorch → NumPy
array = t.numpy()
```

> **Nota:** `torch.from_numpy` comparteix memòria amb l'array de NumPy original. Modificar un modifica l'altre. Si vols una còpia independent usa `torch.tensor(array)`.

## Operacions bàsiques

Algunes operacions bàsiques que poden ser útils, en concret les que ens proporcionen informació:

```python
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

a + b                    # suma element a element
a * b                    # producte element a element
torch.matmul(a, b)       # producte escalar (o matricial)

# Informació del tensor
a.shape                  # dimensions
a.dtype                  # tipus de dades
a.device                 # cpu o cuda
```

## Targeta gràfica
Les GPUs estan dissenyades per efectuar moltes operacions matemàtiques en paral·lel, cosa que les fa molt més eficients que les CPUs per entrenar xarxes neuronals. Les operacions principals d'una xarxa neurolal (multiplicacions de matrius i càlcul de gradients) es beneficien especialment d'aquesta paral·lelització. CUDA és la plataforma de NVIDIA que permet aprofitar la GPU des de PyTorch.
Per saber si tenim una GPU disponible:

```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
```
Per usar la GPU, cal moure tant el model com les dades al mateix dispositiu. És important tenir en compte que el model i les dades han d'estar sempre al mateix dispositiu. Barrejar tensors de CPU i GPU genera un error. Els canvis respecte al codi anterior són mínims:

```python
# Moure el model a la GPU
model = model.to(device)

# Moure les dades a la GPU
X_train = X_train.to(device)
X_test  = X_test.to(device)
y_train = y_train.to(device)
y_test  = y_test.to(device)
```
La resta del codi (training loop, avaluació) no necessita cap canvi, ja que PyTorch gestiona automàticament les operacions al dispositiu corresponent.

## Guardar i carregar models

TODO

Un cop acabat l'entrenament podem emprar la següent comanda per guardar els pesos del model:

```python

torch.save(model.state_dict(), 'model.pth') # podem
```

En qualsevol moment, podem arregar el model. Per carregar els pesos, cal primer instanciar el model amb la mateixa arquitectura i després carregar els pesos:

```python
# Instanciar el model amb la mateixa arquitectura
model = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 3)
)

# Carregar els pesos
model.load_state_dict(torch.load('model.pth'))
model.eval()
```

Per què empram `state_dict`? Un `state_dict` és simplement un diccionari Python que mapeja cada capa del model amb els seus pesos i biaixos. Guardar només els pesos (i no el model sencer) és més robust davant canvis de codi i versions de PyTorch.

## Exercicis

En aquests exercicis tornarem a emprar el mateix conjunt de dades que en el bloc anterior: [Ocean Quality Dataset](https://www.kaggle.com/datasets/vinothkannaece/ocean-quality-dataset)

1. Entrenar un MLP amb la mateixa configuració que Sci-kit però ara emprant Pytorch. Pots triar la tasca 1 (classificació) o la tasca 2 (regressió sobre la variable ZZZZ).
2. Repetir l'entrenament amb diferents valors del ratio d'aprenentatge i visualitzar com canvia l'evolució de la pèrdua. Recorda que aquest paràmetre té sentit amb valors propers a 0. **Extra**: Fer el gràfic emprant la llibreria `matplotlib`.
3. Guardar el millor model. En una altra cel·la o `script` de Ptyhon carregar-lo i fer prediccions sense tornar a entrenar.